# Retinal concept explorer

Generate **cat-style** concept figures one at a time and keep re-rolling until you find a perfect match.

**How to use**
1. Run **Setup** once (loads BiomedCLIP + the 3 classifiers — a few seconds).
2. Run **Generate** repeatedly (`Ctrl`+`Enter`). Each run draws a *new random* fundus image, learns the LAD basis, and shows one image → 3 named concepts.
3. Every figure is auto-saved to `outputs/retinal/cases/` (prefixed `nb_`).
4. To pin a backbone or disease, set `BACKBONE` / `DISEASE` at the top of the Generate cell.


In [ ]:
# === SETUP — run once ===
import random, sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt

# locate repo root (folder with paths.py) so this works wherever Jupyter started
_here = Path.cwd()
REPO = next((p for p in [_here, *_here.parents] if (p / "paths.py").exists()), _here)
sys.path.insert(0, str(REPO))
import paths
from lad.backbones import load_classifier_checkpoint
from lad.clip_concepts import load_clip_model
from lad.retinal import probe_and_learn, render_cat_figure

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP = "biomedclip"
BACKBONES = ["resnet34", "resnet50", "densenet121"]
DISEASES = ["diabetic_retinopathy", "glaucoma", "cataract", "amd"]
CASES_DIR = REPO / "outputs" / "retinal" / "cases"

print(f"device={DEVICE} - loading CLIP '{CLIP}' + {len(BACKBONES)} classifiers ...")
clip_model, _, clip_tok, _ = load_clip_model(CLIP, device=DEVICE)
CLIP_BUNDLE = (clip_model, clip_tok)
CLASSIFIERS = {bb: load_classifier_checkpoint(paths.MODELS_DIR / f"best_{bb}_odir.pt", device=DEVICE)
               for bb in BACKBONES}
print("ready - run the Generate cell below, and re-run it as often as you like.")


In [ ]:
# === GENERATE - re-run me (Ctrl+Enter) for a new random case ===
BACKBONE = None   # pin e.g. "densenet121", or None for random
DISEASE  = None   # pin e.g. "glaucoma", or None for random

bb  = BACKBONE or random.choice(BACKBONES)
cls = DISEASE  or random.choice(DISEASES)
img_dir = paths.RETINAL_FILTERED_ROOT / bb / "correct" / cls
imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
random.shuffle(imgs)
probe = imgs[:15]                      # learn W on 15 random images (fast)

r = probe_and_learn(cls, CLIP, probe, checkpoint=paths.MODELS_DIR / f"best_{bb}_odir.pt",
                    concepts_dir=paths.RETINAL_CONCEPTS_DIR, device=DEVICE, grid=(14, 14),
                    radius=16, epochs=150, classifier=CLASSIFIERS[bb], clip_bundle=CLIP_BUNDLE)

i = random.randrange(len(probe))
S, concepts = r["S_grid"][i], r["concepts"]
top3 = S.sum(dim=(0, 1)).topk(3).indices.tolist()
save_path = CASES_DIR / f"nb_{bb}_{cls}_{probe[i].stem}.png"
fig = render_cat_figure(probe[i], S, top3, [concepts[t] for t in top3],
                        caption=f"{cls.replace('_', ' ')}  ·  {bb}", save_path=save_path)
print(f"{bb} | {cls} | {probe[i].name}  ->  saved {save_path.name}")
plt.show()
